In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

In [ ]:
PROJECT_DIR = Path.cwd().resolve().parent
DATA_DIR    = PROJECT_DIR / 'data'
SRC_DIR     = PROJECT_DIR / 'src'
EXTRACT_DIR = PROJECT_DIR / 'processed' / 'extraction'
REPORT_DIR  = PROJECT_DIR / 'reports'

print('PROJECT_DIR:', PROJECT_DIR)
print('DATA_DIR:', DATA_DIR)
print('SRC_DIR:', SRC_DIR)

In [ ]:
d_items = pd.read_csv(DATA_DIR / 'd_items.csv') # itemid-label 정보
d_labitems = pd.read_csv( DATA_DIR / 'd_labitems.csv') # lab itemid-label 정보

VARIABLE_CATALOG_PATH = SRC_DIR / 'extraction_variable_catalog.csv' # 사용할 변수 카테고리
variable_catalog = pd.read_csv(VARIABLE_CATALOG_PATH)

display(d_items.head())
display(d_labitems.head())
display(variable_catalog.head())

## 1. Cohort

모든 이벤트를 ICU stay 시간 안으로 제한


In [ ]:
patients   = pd.read_csv(DATA_DIR / 'patients.csv')
admissions = pd.read_csv(DATA_DIR / 'admissions.csv')
icustays   = pd.read_csv(DATA_DIR / 'icustays.csv')
services   = pd.read_csv(DATA_DIR / 'services.csv')


In [ ]:
# 입원 및 ICU 입실/퇴실 시간 -> datetime 형식
for col in ['intime', 'outtime']:
    icustays[col] = pd.to_datetime(icustays[col], errors='coerce') # 변환 실패 -> NaT
for col in ['admittime', 'dischtime', 'deathtime']:
    if col in admissions.columns:
        admissions[col] = pd.to_datetime(admissions[col], errors='coerce')
services['transfertime'] = pd.to_datetime(services['transfertime'], errors='coerce')


In [ ]:
# ICU stay 단위로 환자 정보, 입원 정보, ICU 입실 시점 service 정보를 결합
adm_pat_icu = (
    icustays
    .merge(patients[['subject_id', 'gender', 'anchor_age', 'anchor_year', 'anchor_year_group', 'dod']], on='subject_id', how='left')
    .merge(admissions[['subject_id', 'hadm_id', 'admittime', 'dischtime', 'deathtime', 'admission_type', 'race']], on=['subject_id', 'hadm_id'], how='left')
)

specialty_by_stay = (
    services[['subject_id', 'hadm_id', 'transfertime', 'curr_service']]
    .dropna(subset=['curr_service'])
    .merge(
        icustays[['subject_id', 'hadm_id', 'stay_id', 'intime']],
        on=['subject_id', 'hadm_id'],
        how='inner',
    )
    .loc[lambda df: df['transfertime'].le(df['intime'])]
    .sort_values(['stay_id', 'transfertime'])
    .drop_duplicates('stay_id', keep='last')
    [['stay_id', 'curr_service']]
    .rename(columns={'curr_service': 'specialty'})
)
adm_pat_icu = adm_pat_icu.merge(specialty_by_stay, on='stay_id', how='left')

adm_pat_icu['icu_los_hours'] = (adm_pat_icu['outtime'] - adm_pat_icu['intime']).dt.total_seconds() / 3600
display(adm_pat_icu.head())
adm_pat_icu.to_csv(EXTRACT_DIR / 'adm_pat_icu.csv', index=False)


## 2. CHARTEVENTS 추출

In [ ]:
chart_catalog = variable_catalog[variable_catalog['source_table'].eq('chartevents')].copy()
chart_catalog['itemid'] = pd.to_numeric(chart_catalog['itemid'], errors='coerce').astype('int')
chart_catalog.head()

In [ ]:
chart_itemids = chart_catalog['itemid'].tolist()
chart_labels  = chart_catalog['label'].tolist()

# 큰 파일이므로 chunk 단위로 read
chart_chunks = []
chart_cols = ['subject_id', 'hadm_id', 'stay_id', 'charttime', 'itemid', 'value', 'valuenum', 'valueuom']
for chunk in pd.read_csv(DATA_DIR / 'chartevents.csv', usecols=chart_cols, chunksize=1_000_000):
    selected = chunk[chunk['itemid'].isin(chart_itemids)].copy()
    if len(selected):
        selected = selected.merge(d_items[['itemid', 'label', 'category', 'unitname']], on='itemid', how='left')
        selected = selected.merge(chart_catalog[['itemid', 'feature_name', 'type']], on='itemid', how='left')
        chart_chunks.append(selected)

In [ ]:
chart_selected = pd.concat(chart_chunks, ignore_index=True)

chart_selected['charttime'] = pd.to_datetime(chart_selected['charttime'], errors='coerce')
chart_selected = chart_selected.merge(adm_pat_icu[['subject_id', 'hadm_id', 'stay_id', 'intime', 'outtime']],
                                      on=['subject_id', 'hadm_id', 'stay_id'], how='inner')
chart_selected = chart_selected[(chart_selected['charttime'] >= chart_selected['intime']) & 
                                (chart_selected['charttime'] <= chart_selected['outtime'])].copy() # ICU stay 기간 내의 데이터만 남김

chart_selected.to_csv(EXTRACT_DIR / 'chart_selected.csv', index=False)

print('chart_selected rows:', len(chart_selected))
print('chart_selected stays:', chart_selected['stay_id'].nunique())
print('chart_itemid counts to be extracted:', len(chart_catalog))
print('chart_itemid counts extracted:', chart_selected.itemid.nunique())
print('chart_feature counts:', chart_selected.feature_name.nunique())

## 3. LABEVENTS 추출

In [ ]:
lab_catalog = variable_catalog[variable_catalog['source_table'].eq('labevents')].copy()
lab_catalog['itemid'] = pd.to_numeric(lab_catalog['itemid'], errors='coerce').astype('int')
lab_catalog.head()

In [ ]:
lab_itemids = lab_catalog['itemid'].tolist()
lab_labels  = lab_catalog['label'].tolist()

# 큰 파일이므로 chunk 단위로 read
lab_chunks = []
lab_cols = ['labevent_id', 'subject_id', 'hadm_id', 'itemid', 'charttime', 'value', 'valuenum', 'valueuom', 'flag']
for chunk in pd.read_csv(DATA_DIR / 'labevents.csv', usecols=lab_cols, chunksize=1_000_000):
    selected = chunk[chunk['itemid'].isin(lab_itemids)].copy()
    if len(selected):
        selected = selected.merge(d_labitems[['itemid', 'label']], on='itemid', how='left')
        selected = selected.merge(lab_catalog[['itemid', 'feature_name', 'type']], on='itemid', how='left')
        lab_chunks.append(selected)

In [ ]:
lab_selected = pd.concat(lab_chunks, ignore_index=True)

lab_selected['charttime'] = pd.to_datetime(lab_selected['charttime'], errors='coerce')

lab_selected = lab_selected.merge(adm_pat_icu[['subject_id', 'hadm_id', 'stay_id', 'intime', 'outtime']], on=['subject_id', 'hadm_id'], how='inner')
lab_selected = lab_selected[(lab_selected['charttime'] >= lab_selected['intime']) &
                            (lab_selected['charttime'] <= lab_selected['outtime'])].copy()

lab_selected.to_csv(EXTRACT_DIR / 'lab_selected.csv', index=False)

print('lab_selected rows:', len(lab_selected))
print('lab_selected stays:', lab_selected['stay_id'].nunique())
print('lab_itemid counts to be extracted:', len(lab_catalog))
print('lab_itemid counts extracted:', lab_selected.itemid.nunique())
print('lab_feature counts:', lab_selected.feature_name.nunique())


## 4. EMAR 약물 추출


In [ ]:
med_catalog = variable_catalog[variable_catalog['source_table'].eq('emar')].copy()
med_catalog.head()

In [ ]:
med_labels = med_catalog['label'].tolist()
med_catalog['medication_name'] = med_catalog['label'].astype('string').str.strip()

# stopped, not given 등 투약이 이루어지지 않은 경우들도 있으므로 투약한 경우에 해당하는 event_txt들만 추출
administration_event_txt = [
    'Administered', 'Delayed Administered', 'Administered in Other Location',
    'Administered Bolus from IV Drip', 'Partial Administered',
    'Started', 'Delayed Started', 'Started in Other Location', 'Restarted',
    'Applied', 'Applied in Other Location', 'Removed Existing / Applied New',
    'Removed Existing / Applied New in Other Location', 'Confirmed',
    'Confirmed in Other Location', 'Delayed Confirmed', 'Infusion Reconciliation',
    'Rate Change',
]

In [ ]:
emar_cols = ['subject_id', 'hadm_id', 'emar_id', 'emar_seq', 'charttime', 'medication', 'event_txt']
emar = pd.read_csv(DATA_DIR / 'emar.csv', usecols=emar_cols)
emar['charttime'] = pd.to_datetime(emar['charttime'], errors='coerce')
emar['medication_name'] = emar['medication'].astype('string').str.strip()
emar = emar[emar['event_txt'].isin(administration_event_txt)].copy()

In [ ]:
med_selected = emar.merge(med_catalog[['medication_name', 'feature_name']], on='medication_name', how='inner')
med_selected = med_selected.merge(adm_pat_icu[['subject_id', 'hadm_id', 'stay_id', 'intime', 'outtime']], on=['subject_id', 'hadm_id'], how='inner')
med_selected = med_selected[med_selected['charttime'].notna() & 
                            (med_selected['charttime'] >= med_selected['intime']) &
                            (med_selected['charttime'] <= med_selected['outtime'])].copy()

med_selected.to_csv(EXTRACT_DIR / 'medication_events.csv', index=False)

print('medication_events rows:', len(med_selected))
print('medication_events stays:', med_selected['stay_id'].nunique())
print('medication label counts to be extracted:', len(med_labels))
print('medication label counts extracted:', med_selected.medication_name.nunique())
print('medication feature counts:', med_selected.feature_name.nunique())

## 5. PROCEDUREEVENTS 추출


In [ ]:
procedure_catalog = variable_catalog[variable_catalog['source_table'].eq('procedureevents')].copy()
procedure_catalog['itemid'] = pd.to_numeric(procedure_catalog['itemid'], errors='coerce').astype('int')
procedure_catalog.head()

In [ ]:
procedure_itemids = procedure_catalog['itemid'].tolist()
procedure_labels  = procedure_catalog['label'].tolist()

# 큰 파일이므로 chunk 단위로 read
procedure_chunks = []
procedure_cols = ['subject_id', 'hadm_id', 'stay_id', 'starttime', 'endtime', 'itemid', 'value', 'valueuom', 'statusdescription']

for chunk in pd.read_csv(DATA_DIR / 'procedureevents.csv', usecols=procedure_cols, chunksize=500_000):
    selected = chunk[chunk['itemid'].isin(procedure_itemids)].copy()
    if len(selected):
        selected = selected.merge(d_items[['itemid', 'label', 'category']], on='itemid', how='left')
        selected = selected.merge(procedure_catalog[['itemid', 'feature_name', 'type']], on='itemid', how='left')
        procedure_chunks.append(selected)


In [ ]:
procedure_selected = pd.concat(procedure_chunks, ignore_index=True)

procedure_selected['starttime'] = pd.to_datetime(procedure_selected['starttime'], errors='coerce')
procedure_selected['endtime']   = pd.to_datetime(procedure_selected['endtime'], errors='coerce')

procedure_selected = procedure_selected.merge(adm_pat_icu[['subject_id', 'hadm_id', 'stay_id', 'intime', 'outtime']], 
                                              on=['subject_id', 'hadm_id', 'stay_id'], how='inner')
procedure_selected = procedure_selected[(procedure_selected['starttime'] <= procedure_selected['outtime']) &
                                        (procedure_selected['endtime'] >= procedure_selected['intime'])].copy()

procedure_selected.to_csv(EXTRACT_DIR / 'procedure_selected.csv', index=False)

print('procedure_selected rows:', len(procedure_selected))
print('procedure_selected stays:', procedure_selected['stay_id'].nunique())
print('procedure_itemid counts to be extracted:', len(procedure_catalog))
print('procedure_itemid counts extracted:', procedure_selected.itemid.nunique())
print('procedure_feature counts:', procedure_selected.feature_name.nunique())

## 6. Long-format event table 저장

In [ ]:
chart_long = chart_selected.copy()
chart_long['source_table'] = 'chartevents'
chart_long = chart_long[['source_table', 'subject_id', 'hadm_id', 'stay_id', 'charttime', 'itemid', 'label', 'feature_name', 'type', 'value', 'valuenum', 'valueuom']]

lab_long = lab_selected.copy()
lab_long['source_table'] = 'labevents'
lab_long['type'] = 'numeric'
lab_long = lab_long[['source_table', 'subject_id', 'hadm_id', 'stay_id', 'charttime', 'itemid', 'label', 'feature_name', 'type', 'value', 'valuenum', 'valueuom']]

med_long = med_selected.copy()
med_long['source_table'] = 'emar'
med_long['itemid'] = pd.NA
med_long['label'] = med_long['medication_name']
med_long['type'] = 'binary'
med_long['value'] = 1
med_long['valuenum'] = 1
med_long['valueuom'] = pd.NA
med_long = med_long[['source_table', 'subject_id', 'hadm_id', 'stay_id', 'charttime', 'itemid', 'label', 'feature_name', 'type', 'value', 'valuenum', 'valueuom']]

all_events_long = pd.concat([chart_long, lab_long, med_long], ignore_index=True)
all_events_long.to_csv(EXTRACT_DIR / 'all_events_long.csv', index=False)

print('all_events_long rows:', len(all_events_long))
print(all_events_long['source_table'].value_counts())


In [ ]:
all_events_long